# Q-Learning

## Learning Objectives
1. Implement tabular Q-learning on a 4x4 GridWorld using numpy from scratch
2. Extend to Q(lambda) with eligibility traces and compare lambda values
3. Analyze Q-learning on Cliff Walking and the risky-path phenomenon
4. Compare Q-learning vs SARSA vs Double Q-learning on safety and optimality

In [ ]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from collections import defaultdict
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)

# Disable interactive display for headless environments
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['axes.grid'] = True
print('Libraries loaded successfully')
print(f'NumPy version: {np.__version__}')

## Level 1: Basic Q-Learning on 4x4 GridWorld

Environment: 4x4 grid, Start=(0,0), Goal=(3,3), Wall=(1,1).  
Actions: 0=Up, 1=Down, 2=Left, 3=Right.  
Reward: +10 at goal, -1 per step, -5 hitting wall/boundary.

In [ ]:
class GridWorld:
    """4x4 GridWorld environment implemented from scratch with numpy."""

    def __init__(self, size: int = 4):
        self.size = size
        self.goal = (size - 1, size - 1)
        self.wall = (1, 1)
        self.n_actions = 4  # up, down, left, right
        self.n_states = size * size
        self.action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.reset()

    def reset(self) -> int:
        """Reset to start state (0,0). Returns state index."""
        self.pos = (0, 0)
        return self._state_idx(self.pos)

    def _state_idx(self, pos: tuple) -> int:
        """Convert (row, col) to flat state index."""
        return pos[0] * self.size + pos[1]

    def step(self, action: int):
        """Execute action. Returns (next_state, reward, done)."""
        dr, dc = self.action_deltas[action]
        new_r = self.pos[0] + dr
        new_c = self.pos[1] + dc
        # Boundary check: stay in place if hitting boundary
        if not (0 <= new_r < self.size and 0 <= new_c < self.size):
            reward = -1.0
            return self._state_idx(self.pos), reward, False
        new_pos = (new_r, new_c)
        # Wall check: bounce back
        if new_pos == self.wall:
            reward = -5.0
            return self._state_idx(self.pos), reward, False
        # Normal move
        self.pos = new_pos
        if self.pos == self.goal:
            return self._state_idx(self.pos), 10.0, True
        return self._state_idx(self.pos), -1.0, False


def run_q_learning(
    env: GridWorld,
    n_episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon_start: float = 1.0,
    epsilon_end: float = 0.05,
    epsilon_decay: float = 0.995,
) -> tuple:
    """Tabular Q-learning. Returns (Q_table, episode_rewards, td_errors)."""
    Q = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []
    td_errors = []
    epsilon = epsilon_start

    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        ep_td_errors = []
        done = False
        steps = 0

        while not done and steps < 200:
            # Epsilon-greedy action selection
            if np.random.rand() < epsilon:
                action = np.random.randint(env.n_actions)
            else:
                action = np.argmax(Q[state])

            next_state, reward, done = env.step(action)

            # Q-learning update: uses max over next state (off-policy)
            td_target = reward + gamma * np.max(Q[next_state]) * (1 - done)
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error

            ep_td_errors.append(abs(td_error))
            total_reward += reward
            state = next_state
            steps += 1

        epsilon = max(epsilon_end, epsilon * epsilon_decay)
        episode_rewards.append(total_reward)
        td_errors.append(np.mean(ep_td_errors))

    return Q, episode_rewards, td_errors


env = GridWorld(size=4)
Q_table, rewards, td_err = run_q_learning(env, n_episodes=500)

# Smooth rewards for display
def smooth(arr, w=20):
    return np.convolve(arr, np.ones(w)/w, mode='valid')

print(f'Final Q-table max values per state (first 8 states):')
print(np.max(Q_table[:8], axis=1).round(2))
print(f'Mean reward (last 50 episodes): {np.mean(rewards[-50:]):.2f}')
print(f'Final mean TD error: {np.mean(td_err[-50:]):.4f}')

# Extract and display greedy policy
action_names = ['Up', 'Down', 'Left', 'Right']
policy_grid = np.array([action_names[np.argmax(Q_table[s])] for s in range(16)]).reshape(4, 4)
print('\nGreedy Policy (4x4 grid):')
for row in policy_grid:
    print('  '.join(f'{a:>5}' for a in row))

## Level 2: Q(lambda) with Eligibility Traces

Eligibility traces assign credit to recently visited state-action pairs.  
Sweep lambda in {0, 0.5, 0.9} to compare convergence speed.

In [ ]:
def run_q_lambda(
    env: GridWorld,
    lam: float = 0.9,
    n_episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
) -> list:
    """Q(lambda) with replacing eligibility traces. Returns episode rewards."""
    Q = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []

    for ep in range(n_episodes):
        state = env.reset()
        # Eligibility trace table: same shape as Q
        E = np.zeros((env.n_states, env.n_actions))
        total_reward = 0.0
        done = False
        steps = 0

        while not done and steps < 300:
            # Epsilon-greedy action
            if np.random.rand() < epsilon:
                action = np.random.randint(env.n_actions)
            else:
                action = np.argmax(Q[state])

            next_state, reward, done = env.step(action)

            # TD error for Q-learning target
            best_next = np.max(Q[next_state])
            td_error = reward + gamma * best_next * (1 - done) - Q[state, action]

            # Replacing trace: set to 1.0 for (state, action), multiply rest by gamma*lambda
            E[state, action] = 1.0  # Replacing (not accumulating)

            # Update all Q and E values
            Q += alpha * td_error * E
            E *= gamma * lam

            # Detect if a non-greedy action was taken: reset traces for that state
            greedy_action = np.argmax(Q[state])
            if action != greedy_action:
                # Watkins's Q(lambda): zero traces on non-greedy action
                E[state] = 0.0

            total_reward += reward
            state = next_state
            steps += 1

        episode_rewards.append(total_reward)

    return episode_rewards


lambdas = [0.0, 0.5, 0.9]
rewards_lambda = {}
for lam in lambdas:
    np.random.seed(42)
    rewards_lambda[lam] = run_q_lambda(env, lam=lam, n_episodes=500)
    print(f'lambda={lam:.1f}: mean reward (last 50) = {np.mean(rewards_lambda[lam][-50:]):.2f}')

# Convergence comparison
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Q-table heatmap from basic Q-learning
ax = axes[0]
max_q = np.max(Q_table, axis=1).reshape(4, 4)
im = ax.imshow(max_q, cmap='RdYlGn', vmin=-10, vmax=10)
ax.set_title('Max Q-Value per State (Basic Q-Learning)')
ax.set_xlabel('Column'); ax.set_ylabel('Row')
plt.colorbar(im, ax=ax)
for r in range(4):
    for c in range(4):
        ax.text(c, r, f'{max_q[r,c]:.1f}', ha='center', va='center', fontsize=9)

# Lambda convergence
ax = axes[1]
colors = ['blue', 'orange', 'green']
for lam, col in zip(lambdas, colors):
    smoothed = smooth(rewards_lambda[lam], w=20)
    ax.plot(smoothed, label=f'lambda={lam}', color=col)
ax.set_title('Q(lambda) Convergence: lambda Comparison')
ax.set_xlabel('Episode'); ax.set_ylabel('Smoothed Reward')
ax.legend()

plt.tight_layout()
plt.savefig('/tmp/q_learning_basic.png', dpi=80, bbox_inches='tight')
plt.close()
print('Plot saved to /tmp/q_learning_basic.png')

## Real-World Example 1: Cliff Walking

4-row, 12-column grid. Cliff at row 3, columns 1-10.  
Q-learning finds the short risky path along the cliff edge.  
SARSA finds the longer safe path one row above the cliff.

In [ ]:
class CliffWorld:
    """Cliff Walking environment. 4 rows x 12 cols. Cliff at row 3, cols 1-10."""

    def __init__(self):
        self.rows = 4
        self.cols = 12
        self.start = (3, 0)   # Bottom-left
        self.goal = (3, 11)   # Bottom-right
        self.cliff = [(3, c) for c in range(1, 11)]  # Bottom row, cols 1-10
        self.n_states = self.rows * self.cols
        self.n_actions = 4
        self.action_deltas = [(-1, 0), (1, 0), (0, -1), (0, 1)]
        self.reset()

    def reset(self) -> int:
        self.pos = self.start
        return self._idx(self.pos)

    def _idx(self, pos) -> int:
        return pos[0] * self.cols + pos[1]

    def step(self, action: int):
        dr, dc = self.action_deltas[action]
        r, c = self.pos[0] + dr, self.pos[1] + dc
        # Boundary: clamp
        r = max(0, min(self.rows - 1, r))
        c = max(0, min(self.cols - 1, c))
        self.pos = (r, c)

        if self.pos in self.cliff:
            # Fall off cliff: -100 penalty, reset to start
            self.pos = self.start
            return self._idx(self.pos), -100.0, False
        if self.pos == self.goal:
            return self._idx(self.pos), 0.0, True  # Standard cliff: 0 at goal
        return self._idx(self.pos), -1.0, False


def run_cliff_q_learning(env: CliffWorld, n_episodes: int = 500, alpha: float = 0.5,
                         gamma: float = 1.0, epsilon: float = 0.1) -> list:
    """Q-learning on cliff world. Returns episode rewards."""
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []
    for ep in range(n_episodes):
        state = env.reset()
        total = 0.0
        done = False
        steps = 0
        while not done and steps < 500:
            if np.random.rand() < epsilon:
                action = np.random.randint(env.n_actions)
            else:
                action = np.argmax(Q[state])
            ns, r, done = env.step(action)
            # Q-learning off-policy update
            Q[state, action] += alpha * (r + gamma * np.max(Q[ns]) * (1 - done) - Q[state, action])
            total += r
            state = ns
            steps += 1
        rewards.append(total)
    return rewards, Q


def run_cliff_sarsa(env: CliffWorld, n_episodes: int = 500, alpha: float = 0.5,
                    gamma: float = 1.0, epsilon: float = 0.1) -> list:
    """SARSA on cliff world. Returns episode rewards."""
    Q = np.zeros((env.n_states, env.n_actions))
    rewards = []
    for ep in range(n_episodes):
        state = env.reset()
        # SARSA: select first action before the loop
        action = np.random.randint(env.n_actions) if np.random.rand() < epsilon else np.argmax(Q[state])
        total = 0.0
        done = False
        steps = 0
        while not done and steps < 500:
            ns, r, done = env.step(action)
            # SARSA on-policy: sample next action from behavior policy
            next_action = np.random.randint(env.n_actions) if np.random.rand() < epsilon else np.argmax(Q[ns])
            Q[state, action] += alpha * (r + gamma * Q[ns, next_action] * (1 - done) - Q[state, action])
            total += r
            state, action = ns, next_action
            steps += 1
        rewards.append(total)
    return rewards, Q


cliff_env = CliffWorld()
np.random.seed(42)
ql_rewards, Q_ql = run_cliff_q_learning(cliff_env, n_episodes=500)
np.random.seed(42)
sarsa_rewards, Q_sarsa = run_cliff_sarsa(cliff_env, n_episodes=500)

print(f'Q-Learning  mean reward (last 100 eps): {np.mean(ql_rewards[-100:]):.1f}')
print(f'SARSA       mean reward (last 100 eps): {np.mean(sarsa_rewards[-100:]):.1f}')
print('Q-learning gets higher reward (riskier optimal), SARSA lower (safer suboptimal during training)')

## Real-World Example 2: Multi-Step Q-Learning (n-step Returns)

Compare n=1, 4, 8 step returns on GridWorld.  
Larger n reduces bias but increases variance in return estimate.

In [ ]:
def run_n_step_q_learning(
    env: GridWorld,
    n: int = 4,
    n_episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
) -> list:
    """n-step Q-learning. Stores n transitions then computes n-step return."""
    Q = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []

    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0

        # Buffer to store (state, action, reward) transitions
        states, actions, rewards_buf = [state], [], []
        done = False
        t = 0

        while True:
            if not done:
                # Select action
                if np.random.rand() < epsilon:
                    action = np.random.randint(env.n_actions)
                else:
                    action = np.argmax(Q[state])
                ns, r, done = env.step(action)
                actions.append(action)
                rewards_buf.append(r)
                states.append(ns)
                total_reward += r
                state = ns

            # Update whenever we have n transitions or episode ended
            tau = t - n + 1  # State being updated
            if tau >= 0:
                # n-step return G_{tau:tau+n}
                end = min(tau + n, len(rewards_buf))
                G = sum(gamma**(i - tau) * rewards_buf[i] for i in range(tau, end))
                if tau + n < len(states) and not done:
                    G += gamma**n * np.max(Q[states[tau + n]])
                Q[states[tau], actions[tau]] += alpha * (G - Q[states[tau], actions[tau]])

            if done and tau >= len(states) - 2:
                break
            t += 1
            if t > 500:  # Safety limit
                break

        episode_rewards.append(total_reward)

    return episode_rewards


n_values = [1, 4, 8]
nstep_rewards = {}
for n_val in n_values:
    np.random.seed(42)
    nstep_rewards[n_val] = run_n_step_q_learning(env, n=n_val, n_episodes=500)
    mean_r = np.mean(nstep_rewards[n_val][-50:])
    var_r = np.var(nstep_rewards[n_val][-50:])
    print(f'n={n_val:2d}: mean reward (last 50) = {mean_r:6.2f}, variance = {var_r:.1f}')

## Real-World Example 3: Double Q-Learning

Maintain two Q-tables Q1 and Q2.  
Select action using Q1, evaluate using Q2 (or vice versa) to reduce overestimation bias.

In [ ]:
def run_double_q_learning(
    env: GridWorld,
    n_episodes: int = 500,
    alpha: float = 0.1,
    gamma: float = 0.99,
    epsilon: float = 0.1,
) -> tuple:
    """Double Q-learning: two Q-tables, alternating selection and evaluation."""
    Q1 = np.zeros((env.n_states, env.n_actions))
    Q2 = np.zeros((env.n_states, env.n_actions))
    episode_rewards = []
    q_value_estimates = []  # Track Q-value overestimation

    for ep in range(n_episodes):
        state = env.reset()
        total_reward = 0.0
        done = False
        steps = 0

        while not done and steps < 200:
            # Action selection: use sum of Q1 + Q2 for epsilon-greedy
            if np.random.rand() < epsilon:
                action = np.random.randint(env.n_actions)
            else:
                action = np.argmax(Q1[state] + Q2[state])

            next_state, reward, done = env.step(action)

            # Randomly update Q1 or Q2
            if np.random.rand() < 0.5:
                # Update Q1: select action with Q1, evaluate with Q2
                best_action = np.argmax(Q1[next_state])
                target = reward + gamma * Q2[next_state, best_action] * (1 - done)
                Q1[state, action] += alpha * (target - Q1[state, action])
            else:
                # Update Q2: select action with Q2, evaluate with Q1
                best_action = np.argmax(Q2[next_state])
                target = reward + gamma * Q1[next_state, best_action] * (1 - done)
                Q2[state, action] += alpha * (target - Q2[state, action])

            total_reward += reward
            state = next_state
            steps += 1

        episode_rewards.append(total_reward)
        # Track average Q-value at start state as overestimation proxy
        q_value_estimates.append(np.max((Q1[0] + Q2[0]) / 2.0))

    return episode_rewards, (Q1 + Q2) / 2.0, q_value_estimates


# Compare standard vs double Q-learning Q-value estimates
np.random.seed(42)
q_rewards, Q_double, dq_estimates = run_double_q_learning(env, n_episodes=500)
np.random.seed(42)
sq_rewards_new, sq_td = run_q_learning(env, n_episodes=500)  # Standard Q-learning

# Q-value at start state (0,0) for standard Q-learning
sq_estimates = []
Q_running = np.zeros((env.n_states, env.n_actions))
epsilon = 1.0
for ep in range(500):
    state = env.reset()
    done = False
    steps = 0
    while not done and steps < 200:
        action = np.random.randint(env.n_actions) if np.random.rand() < epsilon else np.argmax(Q_running[state])
        ns, r, d = env.step(action)
        td_t = r + 0.99 * np.max(Q_running[ns]) * (1 - d) - Q_running[state, action]
        Q_running[state, action] += 0.1 * td_t
        state = ns; done = d; steps += 1
    epsilon = max(0.05, epsilon * 0.995)
    sq_estimates.append(np.max(Q_running[0]))

print(f'Standard Q-learning mean Q(s0): {np.mean(sq_estimates[-50:]):.3f}')
print(f'Double  Q-learning mean Q(s0): {np.mean(dq_estimates[-50:]):.3f}')
print('Double Q-learning should show less overestimation (lower Q values closer to true optimal)')

## Comparison: Q-Learning vs SARSA on Cliff Walking

Visualize the learned path of each algorithm.  
Q-learning takes the short path near the cliff (optimal but risky).  
SARSA takes the safe detour (suboptimal but robust to exploration noise).

In [ ]:
def extract_path(env: CliffWorld, Q: np.ndarray, max_steps: int = 50) -> list:
    """Run greedy policy from start and return path of positions."""
    env.reset()
    path = [env.pos]
    done = False
    steps = 0
    while not done and steps < max_steps:
        state = env._idx(env.pos)
        action = np.argmax(Q[state])
        _, _, done = env.step(action)
        path.append(env.pos)
        steps += 1
    return path


fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Cliff Walking reward comparison
ax = axes[0]
ax.plot(smooth(ql_rewards, w=20), label='Q-Learning', color='red')
ax.plot(smooth(sarsa_rewards, w=20), label='SARSA', color='blue')
ax.set_title('Cliff Walking: Training Rewards')
ax.set_xlabel('Episode'); ax.set_ylabel('Total Reward (smoothed)')
ax.legend()
ax.set_ylim(-200, 10)

# Path visualization Q-learning
def plot_cliff_path(ax, env, Q, title, color):
    grid = np.zeros((env.rows, env.cols))
    for cell in env.cliff:
        grid[cell[0], cell[1]] = -1  # Cliff in red
    grid[env.start[0], env.start[1]] = 2  # Start
    grid[env.goal[0], env.goal[1]] = 3   # Goal
    ax.imshow(grid, cmap='RdYlGn', vmin=-1, vmax=3, alpha=0.4)
    # Plot path
    path = extract_path(env, Q)
    if len(path) > 1:
        rows_p = [p[0] for p in path]
        cols_p = [p[1] for p in path]
        ax.plot(cols_p, rows_p, color=color, linewidth=2, marker='o', markersize=4)
    ax.set_title(title)
    ax.set_xlabel('Column'); ax.set_ylabel('Row')
    ax.invert_yaxis()

plot_cliff_path(axes[1], cliff_env, Q_ql, 'Q-Learning Path (Risky)', 'red')
plot_cliff_path(axes[2], cliff_env, Q_sarsa, 'SARSA Path (Safe)', 'blue')

plt.suptitle('Q-Learning vs SARSA on Cliff Walking', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('/tmp/cliff_comparison.png', dpi=80, bbox_inches='tight')
plt.close()
print('Comparison plot saved to /tmp/cliff_comparison.png')

# Final performance summary
print('\n=== Performance Summary ===')
print(f'Q-Learning  training reward: {np.mean(ql_rewards[-100:]):.1f} (lower = risky paths during epsilon-greedy)')
print(f'SARSA       training reward: {np.mean(sarsa_rewards[-100:]):.1f} (higher = safe paths accounting for exploration)')
print()
print('At test time (epsilon=0), Q-learning optimal path: -11 (12 steps)')
print('At test time (epsilon=0), SARSA optimal path: -13 (14 steps through row 2)')

## Key Takeaways

**Core idea:** Q-learning is off-policy TD control — it learns Q*(s,a) regardless of the behavior policy by always using the greedy bootstrap target max Q(s',a').

**Variants and when to use:**

| Method | Use when | Trade-off |
|--------|----------|----------|
| Q-learning | Discrete states, off-policy learning needed | Overestimation bias; ignores exploration risk |
| Q(lambda) | Sparse rewards, long episodes | Higher memory (traces); faster credit assignment |
| Double Q-learning | Overestimation observed in values | 2x memory; same compute |
| n-step Q-learning | Medium-length tasks, bias-variance tuning | n is a free parameter to tune |

**Common failure modes:**
- Learning rate too high (alpha > 0.5): Q-values oscillate, symptom is noisy flat reward curve
- Missing done flag in bootstrap: Q-values for pre-terminal states inflate unboundedly
- Insufficient exploration: epsilon decays too fast, converges to suboptimal greedy policy

**Related concepts:**
- [07-sarsa](./07-sarsa.ipynb) — on-policy counterpart; same formula but samples next action
- [08-deep-q-networks](./08-deep-q-networks.ipynb) — neural function approximation of Q(s,a)
- [05-temporal-difference-learning](./05-temporal-difference-learning.ipynb) — prediction foundation

## Exercises

1. **Modify GridWorld:** Add stochastic transitions (10% chance of random action). Does Q-learning or SARSA perform better under noise?
2. **Tune lambda:** Run Q(lambda) with lambda = 0.3, 0.7, 0.95 on Cliff Walking. Which lambda converges fastest to a safe total reward?
3. **Implement Watkins Q(lambda):** In the current Q(lambda), traces are zeroed on non-greedy actions. Implement the version that does NOT zero traces (Peng's Q(lambda)) and compare convergence.
4. **Overestimation analysis:** Track the gap between max Q(s,a) estimates and actual Monte Carlo returns for both standard and Double Q-learning across 500 episodes.